# Phase 4: Cross-Modal Transformer (Real Training V2)

Goal: Train the model using the real ActivityNet features exported from your local machine.

### Requirements:
- Upload the `colab_export` folder to `/content/drive/MyDrive/FedVCMR_Phase4/`


In [ ]:
# Cell 1: Setup & Colab Drive Mount
from google.colab import drive
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm

drive.mount('/content/drive')
!pip install -q open_clip_torch
import open_clip


## 1. Load Real Data Packet
Points to the exported manifest and features in Drive.


In [ ]:
# Cell 2: Path Definitions
EXPORT_ROOT = Path('/content/drive/MyDrive/FedVCMR_Phase4/colab_export')
MANIFEST_PATH = EXPORT_ROOT / 'manifest.json'
FEATURE_DIR = EXPORT_ROOT / 'features'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_PATH}. Please check your Drive upload.")

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f"Loaded {len(manifest)} chunks from manifest.")


## 2. Setup CLIP Backbone
We need the text encoder to process the captions in the manifest.


In [ ]:
# Cell 3: Initialize CLIP for Text Encoding
device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, _, preprocess = open_clip.create_model_and_transforms('MobileCLIP-S1', pretrained='datacompdr')
model_clip = model_clip.to(device)
tokenizer = open_clip.get_tokenizer('MobileCLIP-S1')

@torch.no_grad()
def encode_text(text):
    tokens = tokenizer([text]).to(device)
    text_features = model_clip.encode_text(tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    return text_features.cpu().numpy()


## 3. ActivityNet Dataset
Efficiently loads chunk features and encodes ground-truth boundary targets.


In [ ]:
# Cell 4: ActivityNet Dataset Implementation
class ActivityNetColabDataset(Dataset):
    def __init__(self, manifest, feature_dir):
        self.samples = []
        self.feature_dir = Path(feature_dir)
        
        print("Preprocessing manifest and encoding text (this might take a minute)...")
        for item in tqdm(manifest):
            chunk_id = item['chunk_id']
            t_start = item['t_start']
            t_end = item['t_end']
            duration = t_end - t_start
            
            # Use the first match for training
            if item['matches']:
                match = item['matches'][0]
                caption = match['caption']
                gt_start = match['gt_start']
                gt_end = match['gt_end']
                
                # Normalize GT to chunk-relative 0-1 range
                # Example: Chunk [10, 18], GT [12, 16] -> Start 0.25, End 0.75
                norm_start = max(0.0, (gt_start - t_start) / duration)
                norm_end = min(1.0, (gt_end - t_start) / duration)
                
                if norm_start < norm_end:
                    self.samples.append({
                        'chunk_id': chunk_id,
                        'text': caption,
                        'target': np.array([norm_start, norm_end], dtype=np.float32)
                    })
        
        # Pre-cache text embeddings to speed up training
        self.text_cache = {s['text']: encode_text(s['text'])[0] for s in self.samples}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        feat_path = self.feature_dir / f"{s['chunk_id']}.npy"
        
        # Load vis features (8, 512)
        vis_feat = np.load(feat_path).astype(np.float32)
        # Get text features (512,)
        txt_feat = self.text_cache[s['text']]
        
        return torch.from_numpy(vis_feat), torch.from_numpy(txt_feat), torch.from_numpy(s['target'])

dataset = ActivityNetColabDataset(manifest, FEATURE_DIR)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print(f"Dataset ready with {len(dataset)} training pairs.")


## 4. Transformer Model Architecture
The same 2-layer, 4-head architecture for Milestone 13.


In [ ]:
# Cell 5: Cross-Modal Transformer
class CrossModalTransformer(nn.Module):
    def __init__(self, embed_dim=256, in_dim=512, num_heads=4, num_layers=2):
        super().__init__()
        self.vis_proj = nn.Linear(in_dim, embed_dim)
        self.txt_proj = nn.Linear(in_dim, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True,
            dim_feedforward=embed_dim * 4, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.boundary_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 2),
            nn.Sigmoid()
        )
        
    def forward(self, vis_feats, text_feat):
        # text_feat comes in as (B, 512), make it (B, 1, 512)
        if text_feat.dim() == 2:
            text_feat = text_feat.unsqueeze(1)
            
        v = self.vis_proj(vis_feats)
        t = self.txt_proj(text_feat)
        seq = torch.cat([t, v], dim=1)
        out_seq = self.transformer(seq)
        vis_out = out_seq[:, 1:, :] 
        chunk_rep = vis_out.mean(dim=1)
        return self.boundary_head(chunk_rep)


## 5. Optimized Training Loop (M14)
Now using real data convergence.


In [ ]:
# Cell 6: Training with Adam
model = CrossModalTransformer().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 50
print("Starting real training on ActivityNet features...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for v_batch, t_batch, targets in dataloader:
        v_batch, t_batch, targets = v_batch.to(device), t_batch.to(device), targets.to(device)
        
        optimizer.zero_grad()
        preds = model(v_batch, t_batch)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {total_loss/len(dataloader):.6f}")

# Save the real weights
SAVE_PATH = Path('/content/drive/MyDrive/FedVCMR_Phase4/transformer_best.pt')
torch.save(model.state_dict(), SAVE_PATH)
print(f"Training Complete. Model saved to {SAVE_PATH}")


## Verification (M15 Preview)
Run a single real inference.


In [ ]:
# Cell 7: Final Smoke Test
model.eval()
with torch.no_grad():
    v, t, gt = dataset[0]
    v, t = v.unsqueeze(0).to(device), t.unsqueeze(0).to(device)
    pred = model(v, t).cpu().numpy()[0]
    print(f"GT Boundary: {gt.numpy()}")
    print(f"Pred Boundary: {pred}")
